In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

In [2]:
path = "../dataset/AzureFunctionsInvocationTraceForTwoWeeksJan2021.txt"
df = pd.read_csv(path)

In [3]:

# Compute derived columns
df['arrival_time'] = df['end_timestamp'] - df['duration']
df.sort_values(by='arrival_time', inplace=True)
hour_trace = 4
df['trace_id'] = (df['arrival_time'] // (hour_trace * 3600)).astype(int)
df['arrival_time_norm'] = df['arrival_time'] - (df['trace_id'] * hour_trace * 3600)

In [4]:
df

,app,func,end_timestamp,duration,arrival_time,trace_id,arrival_time_norm
0,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,e3cdb48830f66eb8689cc0223514569a69812b77e6611e...,7.949090e-02,0.078,1.490900e-03,0,0.001491
1,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,337cd24a7d5fd5c92460faee4ebe6a186a0eb322bd17b7...,5.715786e+01,57.154,3.860041e-03,0,0.003860
2,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,48cc770d590d3c5a7691b3b4e9302f82ec3be5ddc2a037...,5.913048e+01,59.125,5.477905e-03,0,0.005478
3,f274d71de386ccc77e4ca74766dbc485461c3053059d47...,3d2aee54a133509f16fb636d74128c2adcfcac71c6dcef...,6.252541e+00,6.236,1.654107e-02,0,0.016541
4,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,68bbfd828223a505d7917339f4656c5f33ff93225cdb9d...,6.682396e-02,0.050,1.682396e-02,0,0.016824
...,...,...,...,...,...,...,...
1980946,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209597e+06,0.001,1.209597e+06,83,14396.700017
1980947,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209598e+06,0.001,1.209598e+06,83,14398.190409
1980948,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,83,14398.560519
1980949,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,83,14398.627382


In [6]:
cold_start_threshold = 600  # 10 minutes in seconds
# Sort the dataframe for safety
df = df.sort_values(by=['trace_id', 'arrival_time_norm'])

# Compute inter-arrival time differences within each trace_id
df['prev_arrival_time'] = df.groupby('trace_id')['arrival_time_norm'].shift(1)
df['gap'] = df['arrival_time'] - df['prev_arrival_time']

# Identify trace_ids where all gaps are above threshold or first arrival (NaN) is ignored
valid_trace_ids = (
    df.groupby('trace_id')['gap']
      .apply(lambda gaps: gaps.dropna().gt(cold_start_threshold).all())
)

# Filter trace_ids where all invocations suffer cold start
cold_start_trace_ids = valid_trace_ids[valid_trace_ids].index.tolist()

# Create filtered dataframe
df_cold_start_only = df[df['trace_id'].isin(cold_start_trace_ids)].drop(columns=['prev_arrival_time', 'gap'])

print(f"Selected {len(cold_start_trace_ids)} traces with all cold starts from {df['trace_id'].nunique()} total traces.")


Selected 83 traces with all cold starts from 84 total traces.


In [17]:
import numpy as np

# Parameters
T_max = 14400               # 4 hours in seconds
T_cold = 600                # 10 min cold start threshold
lambda_rate = 1 / 500       # Average 1 invocation every 15 min

inter_arrival_times = []
total_time = 0

while total_time < T_max:
    t = np.random.exponential(1 / lambda_rate)
    if t >= T_cold:
        if total_time + t > T_max:
            break
        inter_arrival_times.append(t)
        total_time += t

# Compute absolute arrival times
arrival_times = np.cumsum(inter_arrival_times)

print(f"Generated {len(arrival_times)} cold-start-only invocations within 4 hours.")
print("Arrival times (s):", arrival_times)

# Optionally, save to CSV for direct replay
import pandas as pd
df = pd.DataFrame({'arrival_time': arrival_times})
# df.to_csv('poisson_coldstart_trace.csv', index=False)

# import ace_tools as tools; tools.display_dataframe_to_user(name="Poisson Cold Start Trace", dataframe=df)
df.shape

Generated 14 cold-start-only invocations within 4 hours.
Arrival times (s): [  966.98110657  2266.68696339  2934.49124118  3949.53503917
  4661.4459538   5455.70930998  6741.61084894  8120.29696335
  9355.7215393  10339.31628965 11052.03885765 12584.68492739
 13384.21471725 14293.59113994]


(14, 1)